# Detecting Cybersecurity Threats Using Deep Learning

This notebook trains a PyTorch binary classifier on preprocessed BETH cybersecurity log data. The target `sus_label` identifies suspicious (`1`) versus benign (`0`) events.


## Dataset

Features: `processId`, `threadId`, `parentProcessId`, `userId`, `mountNamespace`, `argsNum`, and `returnValue`.

Target: `sus_label`.

See `accreditation.md` for the original BETH dataset source and research paper.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim


In [ ]:
# Load preprocessed BETH data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

train_df.head()


In [ ]:
# Separate features and labels
X_train = train_df.drop('sus_label', axis=1)
y_train = train_df['sus_label']

X_val = val_df.drop('sus_label', axis=1)
y_val = val_df['sus_label']

X_test = test_df.drop('sus_label', axis=1)
y_test = test_df['sus_label']

# Fit scaler only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Convert arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(X_train_tensor.shape[1], 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

torch.manual_seed(42)

model = Net()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
# Train for exactly 10 epochs
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{epochs}, Loss: {average_loss:.4f}')


In [ ]:
# Validation accuracy
model.eval()

with torch.no_grad():
    val_outputs = model(X_val_tensor)
    val_probabilities = torch.sigmoid(val_outputs)
    val_predictions = (val_probabilities >= 0.5).float()
    val_accuracy = (val_predictions == y_val_tensor).float().mean().item()

print(f'Validation Accuracy: {val_accuracy:.4f}')

# Test accuracy
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test_outputs)
    test_predictions = (test_probabilities >= 0.5).float()
    test_accuracy = (test_predictions == y_test_tensor).float().mean().item()

print(f'Test Accuracy: {test_accuracy:.4f}')


## Saved run results and limitation

The final DataLab run completed 10 epochs with validation accuracy `1.0000` and test accuracy `0.1554`. The data splits are highly imbalanced and have substantially different class distributions, so raw accuracy should be interpreted cautiously.

Future work should include precision, recall, F1 score, ROC-AUC/PR-AUC, a confusion matrix, class weighting or resampling, threshold tuning, and analysis of distribution shift.
